In [ ]:
# define the state 

from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage


class Store_message(TypedDict):
    """State to store messages."""

    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str = ""
    job_description: str =""
    url:str=""
    md_formated: str =""
    score: float = 0.0
    valid: bool = True

: 

In [ ]:
import argparse
import markdown
import os
import subprocess
import tempfile

# ATS-friendly resume template with professional styling
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{name} - Resume</title>
    <style>
        @page {
            margin: 1cm;
        }
        body {{
            font-family: 'Arial', 'Helvetica', sans-serif;
            line-height: 1.5;
            margin: 0;
            padding: 20px;
            color: #333;
            font-size: 12px;
        }}
        .container {{
            max-width: 800px;
            margin: 0 auto;
        }}
        h1, h2, h3, h4 {{
            color: #2c3e50;
            margin-top: 20px;
            margin-bottom: 10px;
        }}
        h1 {{
            font-size: 28px;
            text-align: center;
            border-bottom: 1px solid #ddd;
            padding-bottom: 10px;
            margin-top: 0;
        }}
        h2 {{
            font-size: 20px;
            border-bottom: 1px solid #eee;
            padding-bottom: 5px;
        }}
        h3 {{
            font-size: 16px;
        }}
        .summary {{
            margin: 15px 0;
        }}
        ul {{
            margin-top: 5px;
            padding-left: 20px;
        }}
        li {{
            margin-bottom: 5px;
        }}
        .contact-info {{
            text-align: center;
            margin-bottom: 20px;
        }}
        .skills {{
            display: flex;
            flex-wrap: wrap;
            margin-top: 5px;
        }}
        .skill {{
            background-color: #f0f7fc;
            padding: 5px 10px;
            margin: 3px;
            border-radius: 3px;
            display: inline-block;
        }}
        .experience-item, .education-item, .project-item {{
            margin-bottom: 15px;
        }}
        .job-title, .degree, .project-title {{
            font-weight: bold;
        }}
        .company, .institution, .project-desc {{
            font-style: italic;
        }}
        .period {{
            float: right;
            color: #666;
        }}
    </style>
</head>
<body>
    <div class="container">
        {content}
    </div>
</body>
</html>
"""

def extract_name(content):
    """Attempt to extract the name from the markdown content"""
    lines = content.split('\n')
    for line in lines:
        if line.strip().startswith('# '):
            return line.strip()[2:].strip()
    return "Professional Resume"

def convert_md_to_html(md_content):
    """Convert markdown content to HTML"""
    html_content = markdown.markdown(md_content)
    name = extract_name(md_content)
    return HTML_TEMPLATE.format(name=name, content=html_content)

def convert_html_to_pdf(html_content, output_path):
    """Convert HTML content to PDF using wkhtmltopdf"""
    with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as temp:
        temp.write(html_content.encode('utf-8'))
        temp_html_path = temp.name
    
    try:
        # Run wkhtmltopdf command
        subprocess.run([
            'wkhtmltopdf',
            '--enable-local-file-access',
            '--margin-top', '10mm',
            '--margin-bottom', '10mm',
            '--margin-left', '10mm',
            '--margin-right', '10mm',
            '--page-size', 'Letter',
            temp_html_path,
            output_path
        ], check=True)
        print(f"PDF successfully created at: {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error creating PDF: {e}")
    except FileNotFoundError:
        print("Error: wkhtmltopdf not found. Please install it first.")
    finally:
        # Clean up the temporary file
        if os.path.exists(temp_html_path):
            os.unlink(temp_html_path)

def main():
    parser = argparse.ArgumentParser(description='Convert Markdown resume to PDF')
    parser.add_argument('input_file', help='Input Markdown file')
    parser.add_argument('-o', '--output', default='resume.pdf', help='Output PDF file (default: resume.pdf)')
    
    args = parser.parse_args()
    
    try:
        with open(args.input_file, 'r', encoding='utf-8') as f:
            md_content = f.read()
        
        html_content = convert_md_to_html(md_content)
        convert_html_to_pdf(html_content, args.output)
        
    except FileNotFoundError:
        print(f"Error: Input file '{args.input_file}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == '__main__':
    main()